# 🏋️ Step 2: ペルソナ別 PPO 学習 → ONNX生成

```
Step1で生成した persona_rewards.json を読み込み
  ↓
ペルソナA〜Eを順番にPPO学習 (各~60M steps)
  ↓
persona_A.onnx, persona_B.onnx ... を生成
  ↓
Step3のビジュアライザーで読み込む
```

**Runtime: T4 GPU**

In [ ]:
!pip install torch onnx onnxscript onnxruntime numpy -q
print('✓ Install complete')

In [ ]:
# セル2: インポート & GPU確認
import torch, torch.nn as nn
import numpy as np
import json, random, math, time, os, subprocess
from IPython.display import clear_output
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams.update({
    'figure.facecolor':'#0a0c10','axes.facecolor':'#12161e',
    'text.color':'#c8d8e8','axes.labelcolor':'#c8d8e8',
    'xtick.color':'#4a5a6a','ytick.color':'#4a5a6a',
})

assert torch.cuda.is_available(), 'GPU未検出'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')

def get_gpu_stats():
    try:
        r=subprocess.run(
            ['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total',
             '--format=csv,noheader,nounits'],
            capture_output=True,text=True,timeout=2)
        v=r.stdout.strip().split(', ')
        return float(v[0]),float(v[1])/1024,float(v[2])/1024
    except: return 0.0,0.0,15.0

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/mesa_persona'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save dir: {SAVE_DIR}')

In [ ]:
# セル3: 定数 & マップ生成
OTHER=0; ROAD=1; BUILDING=2; TREE=3
GRID=30; N_ENVS=4096; MAX_STEPS=400; ROLLOUT=128
EPOCHS=4; CLIP_EPS=0.2; GAE_LAM=0.95; GAMMA=0.99
ENT_COEF=0.03; VF_COEF=0.5; LR=2e-4
MINIBATCH=16384; B=N_ENVS*ROLLOUT
STEPS_PER_PERSONA = 60_000_000

MOVE_DIST=0.25; ROT_RAD=math.radians(20)
RAY_DEG=[-60,-30,0,30,60]; N_RAYS=5
RAY_MAX=6.0; N_STEPS=60; RAY_STEP=RAY_MAX/N_STEPS
RAY_D_T =torch.linspace(RAY_STEP,RAY_MAX,N_STEPS,device=DEVICE)
RAY_DA_T=torch.tensor([math.radians(a) for a in RAY_DEG],
                       dtype=torch.float32,device=DEVICE)
OBS_DIM=N_RAYS*2+8; ACT_DIM=3

# マップ
def make_map(size=GRID,seed=42):
    rng=random.Random(seed)
    g=np.full((size,size),OTHER,np.int32)
    step=4; rows=list(range(0,size,step)); cols=list(range(0,size,step))
    for r in rows: g[r,:]=ROAD
    for c in cols: g[:,c]=ROAD
    for ri in range(len(rows)-1):
        for ci in range(len(cols)-1):
            r0,r1=rows[ri]+1,rows[ri+1]; c0,c1=cols[ci]+1,cols[ci+1]
            cells=[(r,c) for r in range(r0,r1) for c in range(c0,c1)]
            if not cells: continue
            b=rng.choice(cells); g[b[0],b[1]]=BUILDING
            for r,c in cells:
                if (r,c)==b: continue
                v=rng.random()
                if v<.25: g[r,c]=TREE
                elif v<.45: g[r,c]=BUILDING
    for r in rows: g[r,:]=ROAD
    for c in cols: g[:,c]=ROAD
    return g

MAP_NP  =make_map()
MAP_T   =torch.tensor(MAP_NP,dtype=torch.long,device=DEVICE)
MAP_F   =MAP_T.float()
PASS_T  =(MAP_T==ROAD)|(MAP_T==BUILDING)
BCELLS_NP=np.array([(r,c) for r in range(GRID) for c in range(GRID) if MAP_NP[r,c]==BUILDING])
BCELLS_T =torch.tensor(BCELLS_NP,dtype=torch.float32,device=DEVICE)
NB=len(BCELLS_NP)
print(f'B:{B:,}  Buildings:{NB}  OBS:{OBS_DIM}')

In [ ]:
# セル4: レイキャスト (JIT)
@torch.jit.script
def raycast_gpu(
    xs:torch.Tensor,ys:torch.Tensor,ths:torch.Tensor,
    ray_da:torch.Tensor,ray_d:torch.Tensor,
    map_t:torch.Tensor,grid:int,road:int,
)->tuple[torch.Tensor,torch.Tensor]:
    angles=ths.unsqueeze(1)+ray_da.unsqueeze(0)
    dx=torch.cos(angles); dy=torch.sin(angles)
    px=xs[:,None,None]+dx[:,:,None]*ray_d[None,None,:]
    py=ys[:,None,None]+dy[:,:,None]*ray_d[None,None,:]
    in_b=(px>=0.0)&(px<float(grid))&(py>=0.0)&(py<float(grid))
    ri=px.long().clamp(0,grid-1); ci=py.long().clamp(0,grid-1)
    ct=map_t[ri,ci]; ct=torch.where(in_b,ct,torch.zeros_like(ct))
    is_hit=ct!=road
    first=torch.argmax(is_hit.long(),dim=2)
    has_h=is_hit.any(dim=2)
    first=torch.where(has_h,first,torch.full_like(first,ray_d.shape[0]-1))
    hit_ct=ct.gather(2,first.unsqueeze(2)).squeeze(2).float()
    hit_d=ray_d[first]
    hit_ct=torch.where(has_h,hit_ct,torch.ones_like(hit_ct)*float(road))
    return hit_ct/3.0,(hit_d/ray_d[-1]).clamp(0.0,1.0)

# warm-up
_x=torch.rand(N_ENVS,device=DEVICE)*GRID
_y=torch.rand(N_ENVS,device=DEVICE)*GRID
_t=torch.rand(N_ENVS,device=DEVICE)*math.pi*2
for _ in range(3): raycast_gpu(_x,_y,_t,RAY_DA_T,RAY_D_T,MAP_T,GRID,ROAD)
del _x,_y,_t
print('raycast JIT ✓')

In [ ]:
# セル5: ペルソナ対応ベクトル化環境
# 報酬パラメータをコンストラクタで受け取る設計

class PersonaVecEnv:
    """
    ペルソナの報酬パラメータに基づいて報酬が変わる並列環境。
    visited_map で訪問済みセルを記録し explore_bonus / revisit_penalty を計算。
    """
    def __init__(self, n, reward_params):
        self.n  = n
        self.rp = reward_params  # 報酬パラメータ dict

        # 状態テンソル (GPU)
        self.x =torch.zeros(n,device=DEVICE); self.y =torch.zeros(n,device=DEVICE)
        self.th=torch.zeros(n,device=DEVICE)
        self.gx=torch.zeros(n,device=DEVICE); self.gy=torch.zeros(n,device=DEVICE)
        self.pd=torch.zeros(n,device=DEVICE)
        self.stall=torch.zeros(n,dtype=torch.long,device=DEVICE)
        self.scnt =torch.zeros(n,dtype=torch.long,device=DEVICE)
        self.px=torch.zeros(n,device=DEVICE); self.py=torch.zeros(n,device=DEVICE)

        # 訪問済みセルマップ: (N_ENVS, GRID, GRID) bool
        # メモリ節約のためuint8で保持
        self.visited=torch.zeros(n,GRID,GRID,dtype=torch.uint8,device=DEVICE)

        # 他エージェント位置 (social_bonus用): 前stepの位置を参照
        self.other_x=torch.zeros(n,device=DEVICE)
        self.other_y=torch.zeros(n,device=DEVICE)

        self._rst(torch.ones(n,dtype=torch.bool,device=DEVICE))

    def _rand_b(self,mask):
        idx=torch.randint(0,NB,(self.n,),device=DEVICE)
        return BCELLS_T[idx,0]+0.5, BCELLS_T[idx,1]+0.5

    def _rst(self,mask):
        bx,by=self._rand_b(mask); gx,gy=self._rand_b(mask)
        th=torch.rand(self.n,device=DEVICE)*math.pi*2
        self.x =torch.where(mask,bx,self.x);  self.y =torch.where(mask,by,self.y)
        self.th=torch.where(mask,th,self.th)
        self.gx=torch.where(mask,gx,self.gx); self.gy=torch.where(mask,gy,self.gy)
        nd=torch.sqrt((bx-gx)**2+(by-gy)**2)
        self.pd   =torch.where(mask,nd,self.pd)
        self.stall=torch.where(mask,torch.zeros_like(self.stall),self.stall)
        self.scnt =torch.where(mask,torch.zeros_like(self.scnt), self.scnt)
        self.px=torch.where(mask,bx,self.px); self.py=torch.where(mask,by,self.py)
        # 訪問マップをリセット
        self.visited[mask]=0

    def reset_all(self):
        self._rst(torch.ones(self.n,dtype=torch.bool,device=DEVICE))
        return self._obs()

    def _obs(self):
        ht,hd=raycast_gpu(self.x,self.y,self.th,RAY_DA_T,RAY_D_T,MAP_T,GRID,ROAD)
        dx=self.gx-self.x; dy=self.gy-self.y
        ra=torch.atan2(dy,dx)-self.th
        dn=(torch.sqrt(dx**2+dy**2)/(GRID*math.sqrt(2))).clamp(0,1)
        ri=self.x.long().clamp(0,GRID-1); ci=self.y.long().clamp(0,GRID-1)
        cur=MAP_F[ri,ci]/3.0
        return torch.stack([
            ht[:,0],ht[:,1],ht[:,2],ht[:,3],ht[:,4],
            hd[:,0],hd[:,1],hd[:,2],hd[:,3],hd[:,4],
            torch.sin(ra),torch.cos(ra),dn,
            torch.sin(self.th),torch.cos(self.th),cur,
            (self.stall.float()/10).clamp(0,1),
            (self.scnt.float()/MAX_STEPS).clamp(0,1),
        ],dim=1).clamp(-1,1)

    def step(self, actions):
        rp = self.rp
        rew=torch.full((self.n,),-rp['step_penalty'],device=DEVICE)
        self.px=self.x.clone(); self.py=self.y.clone()

        # 回転
        rot=torch.where(actions==1,
              torch.full_like(self.th,-ROT_RAD),
            torch.where(actions==2,
              torch.full_like(self.th, ROT_RAD),
              torch.zeros_like(self.th)))
        self.th=(self.th+rot)%(math.pi*2)

        # 前進
        fwd=(actions==0)
        nx=(self.x+torch.cos(self.th)*MOVE_DIST).clamp(0.01,GRID-0.01)
        ny=(self.y+torch.sin(self.th)*MOVE_DIST).clamp(0.01,GRID-0.01)
        ri=nx.long().clamp(0,GRID-1); ci=ny.long().clamp(0,GRID-1)
        ok=fwd&PASS_T[ri,ci]; bad=fwd&~PASS_T[ri,ci]
        self.x=torch.where(ok,nx,self.x); self.y=torch.where(ok,ny,self.y)

        # 現在セル
        ri2=self.x.long().clamp(0,GRID-1); ci2=self.y.long().clamp(0,GRID-1)
        cur_cell=MAP_T[ri2,ci2]

        # 道路ボーナス
        on_road=ok&(cur_cell==ROAD)
        rew=torch.where(on_road,rew+rp['road_bonus'],rew)

        # 建物ボーナス (毎ステップ)
        on_build=(cur_cell==BUILDING)
        rew=torch.where(on_build,rew+rp['building_bonus'],rew)

        # 前進バイアス
        rew=torch.where(fwd,rew+rp['forward_bias'],rew)

        # 違反ペナルティ
        rew=torch.where(bad,rew-rp['violation_penalty'],rew)

        # 探索ボーナス / 再訪ペナルティ
        env_idx=torch.arange(self.n,device=DEVICE)
        was_visited=self.visited[env_idx,ri2,ci2].bool()
        is_new=ok&~was_visited
        is_revisit=ok&was_visited
        rew=torch.where(is_new,   rew+rp['explore_bonus'],  rew)
        rew=torch.where(is_revisit,rew-rp['revisit_penalty'],rew)
        # 訪問済みに追加
        self.visited[env_idx,ri2,ci2]=1

        # 社交ボーナス (他エージェントとの近接)
        # ← シフトした別環境の位置と比較 (同バッチ内の隣環境を他者とみなす)
        if rp['social_bonus'] > 0:
            ox=torch.roll(self.x,1); oy=torch.roll(self.y,1)
            dist_other=torch.sqrt((self.x-ox)**2+(self.y-oy)**2)
            near=(dist_other<2.0)&ok
            rew=torch.where(near,rew+rp['social_bonus'],rew)

        # 滞留ペナルティ
        moved=(torch.abs(self.x-self.px)+torch.abs(self.y-self.py))>0.05
        self.stall=torch.where(moved,torch.zeros_like(self.stall),self.stall+1)
        rew=torch.where(self.stall>3,rew-rp['stall_penalty'],rew)

        # 距離報酬
        nd=torch.sqrt((self.x-self.gx)**2+(self.y-self.gy)**2)
        rew=torch.where(nd<self.pd,rew+0.5,rew)
        rew=torch.where(nd>self.pd,rew-0.2,rew)
        self.pd=nd

        # ゴール到達
        arr=(cur_cell==BUILDING)&(nd<0.8)
        rew=torch.where(arr,rew+rp['goal_reward'],rew)
        ngx,ngy=self._rand_b(arr)
        self.gx=torch.where(arr,ngx,self.gx); self.gy=torch.where(arr,ngy,self.gy)
        self.pd=torch.where(arr,
            torch.sqrt((self.x-self.gx)**2+(self.y-self.gy)**2),self.pd)

        self.scnt+=1
        done=self.scnt>=MAX_STEPS
        self._rst(done)
        return self._obs(),rew,done

print('PersonaVecEnv ✓')

In [ ]:
# セル6: ポリシーネットワーク (全ペルソナ共通アーキテクチャ)
class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.ray_enc=nn.Sequential(
            nn.Linear(N_RAYS*2,128),nn.LayerNorm(128),nn.ReLU(),
            nn.Linear(128,128),nn.ReLU())
        self.nav_enc=nn.Sequential(nn.Linear(8,64),nn.ReLU())
        self.shared=nn.Sequential(
            nn.Linear(192,512),nn.LayerNorm(512),nn.ReLU(),
            nn.Linear(512,512),nn.LayerNorm(512),nn.ReLU(),
            nn.Linear(512,256),nn.ReLU())
        self.actor =nn.Linear(256,ACT_DIM)
        self.critic=nn.Linear(256,1)

    def forward(self,x):
        h=torch.cat([self.ray_enc(x[:,:N_RAYS*2]),
                     self.nav_enc(x[:,N_RAYS*2:])],dim=-1)
        h=self.shared(h); return self.actor(h),self.critic(h)

    def act(self,x):
        lg,val=self.forward(x)
        d=torch.distributions.Categorical(torch.softmax(lg,-1))
        a=d.sample(); return a,d.log_prob(a),d.entropy(),val.squeeze(-1)

print(f'PolicyNet params: {sum(p.numel() for p in PolicyNet().parameters()):,}')

In [ ]:
# セル7: 1ペルソナ分の学習関数
def train_persona(persona_id, reward_params, total_steps=STEPS_PER_PERSONA):
    print('='*60)
    print(f'学習開始: ペルソナ {persona_id} — {reward_params["persona_name"]}')
    print(f'  {reward_params["description"]}')
    print('='*60)

    policy    = PolicyNet().to(DEVICE)
    optimizer = torch.optim.Adam(policy.parameters(),lr=LR)
    env       = PersonaVecEnv(N_ENVS, reward_params)
    obs       = env.reset_all()

    # チェックポイント再開確認
    ckpt_path = f'{SAVE_DIR}/ckpt_{persona_id}.pt'
    start_update = 0
    log = {'reward':[],'trips':[],'viols':[],'explore':[]}
    if os.path.exists(ckpt_path):
        ck=torch.load(ckpt_path,map_location=DEVICE)
        policy.load_state_dict(ck['policy'])
        optimizer.load_state_dict(ck['optimizer'])
        start_update=ck.get('update',0)
        log=ck.get('log',log)
        print(f'  ✓ Resumed from update={start_update}')

    # バッファ
    obs_buf =torch.zeros(ROLLOUT,N_ENVS,OBS_DIM,device=DEVICE)
    act_buf =torch.zeros(ROLLOUT,N_ENVS,dtype=torch.long,device=DEVICE)
    rew_buf =torch.zeros(ROLLOUT,N_ENVS,device=DEVICE)
    done_buf=torch.zeros(ROLLOUT,N_ENVS,device=DEVICE)
    logp_buf=torch.zeros(ROLLOUT,N_ENVS,device=DEVICE)
    val_buf =torch.zeros(ROLLOUT,N_ENVS,device=DEVICE)

    total=start_update*B; r_s=t_s=v_s=e_s=0.0; t0=time.time()
    LOG_EVERY=10

    for update in range(start_update, total_steps//B+1):
        # Rollout
        for t in range(ROLLOUT):
            with torch.no_grad():
                actions,logps,_,values=policy.act(obs)
            obs_buf[t]=obs; act_buf[t]=actions
            logp_buf[t]=logps; val_buf[t]=values
            obs,rew,done=env.step(actions)
            rew_buf[t]=rew; done_buf[t]=done.float()
            total+=N_ENVS
            r_s+=rew.sum().item()
            t_s+=(rew>=reward_params['goal_reward']*0.8).sum().item()
            v_s+=(rew<=-reward_params['violation_penalty']*0.8).sum().item()
            e_s+=(rew>=reward_params['explore_bonus']*0.8).sum().item()

        # GAE
        with torch.no_grad():
            _,lv=policy.forward(obs); lv=lv.squeeze(-1)
        adv=torch.zeros_like(rew_buf); gae=torch.zeros(N_ENVS,device=DEVICE)
        for t in reversed(range(ROLLOUT)):
            nv=lv if t==ROLLOUT-1 else val_buf[t+1]
            delta=rew_buf[t]+GAMMA*nv*(1-done_buf[t])-val_buf[t]
            gae=delta+GAMMA*GAE_LAM*(1-done_buf[t])*gae; adv[t]=gae
        ret=adv+val_buf

        # PPO更新
        obs_f=obs_buf.reshape(B,OBS_DIM); act_f=act_buf.reshape(B)
        logp_f=logp_buf.reshape(B)
        adv_f=adv.reshape(B); ret_f=ret.reshape(B)
        adv_f=(adv_f-adv_f.mean())/(adv_f.std()+1e-8)
        for _ in range(EPOCHS):
            perm=torch.randperm(B,device=DEVICE)
            for s in range(0,B,MINIBATCH):
                mb=perm[s:s+MINIBATCH]
                lg,vs=policy.forward(obs_f[mb])
                d=torch.distributions.Categorical(torch.softmax(lg,-1))
                nlp=d.log_prob(act_f[mb]); ent=d.entropy().mean()
                ratio=torch.exp(nlp-logp_f[mb])
                lpi=-torch.min(ratio*adv_f[mb],
                    torch.clamp(ratio,1-CLIP_EPS,1+CLIP_EPS)*adv_f[mb]).mean()
                lvf=(vs.squeeze()-ret_f[mb]).pow(2).mean()
                loss=lpi+VF_COEF*lvf-ENT_COEF*ent
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(),0.5)
                optimizer.step()

        if update%LOG_EVERY==0 and update>0:
            d=LOG_EVERY*B
            ar=r_s/d; at=t_s/d*100; av=v_s/d*100; ae=e_s/d*100
            gpu_util,gpu_used,gpu_total=get_gpu_stats()
            sps=LOG_EVERY*B/(time.time()-t0)
            log['reward'].append(round(float(ar),4))
            log['trips'].append(round(float(at),3))
            log['viols'].append(round(float(av),3))
            log['explore'].append(round(float(ae),3))
            r_s=t_s=v_s=e_s=0.0; t0=time.time()
            clear_output(wait=True)
            # 学習曲線
            fig,axes=plt.subplots(1,4,figsize=(16,3))
            fig.patch.set_facecolor('#0a0c10')
            for ax,(k,c,t_) in zip(axes,[
                ('reward','#00ddb4','Reward/step'),
                ('trips','#ffc840','GoalRate%'),
                ('viols','#ff5050','ViolRate%'),
                ('explore','#aa88ff','ExploreRate%'),
            ]):
                v=log[k]
                if v: ax.plot(v,color=c,lw=1.5)
                ax.set_title(f'{t_} {v[-1]:.3f}' if v else t_,
                             color='#c8d8e8',fontsize=9)
                ax.grid(color='#1e2530',lw=.5)
                ax.spines[:].set_color('#1e2530')
            name=reward_params['persona_name']
            fig.suptitle(f'[{persona_id}] {name} | Upd:{update:,} | Steps:{total/1e6:.1f}M',
                         color='#00ddb4',fontsize=11)
            plt.tight_layout(); plt.show()
            print(f'[{persona_id}] Upd:{update:5d} | R:{ar:.4f} | '
                  f'Goal:{at:.2f}% | Viol:{av:.2f}% | Explore:{ae:.2f}% | '
                  f'{sps:.0f}sps | GPU:{gpu_util:.0f}% VRAM:{gpu_used:.2f}GB')

        if update%50==0 and update>0:
            torch.save({'policy':policy.state_dict(),
                        'optimizer':optimizer.state_dict(),
                        'update':update,'log':log}, ckpt_path)
            print(f'  💾 [{persona_id}] Saved')

    print(f'\n[{persona_id}] 学習完了')
    return policy, log

print('train_persona ✓')

In [ ]:
# セル8: ONNX エクスポート関数
def export_persona_onnx(policy, persona_id, reward_params):
    policy.eval().to('cpu')
    class ActorOnly(nn.Module):
        def __init__(self,p): super().__init__(); self.p=p
        def forward(self,x): l,_=self.p(x); return l
    path = f'{SAVE_DIR}/persona_{persona_id}.onnx'
    torch.onnx.export(
        ActorOnly(policy), torch.zeros(1,OBS_DIM), path,
        input_names=['state'],output_names=['logits'],
        dynamic_axes={'state':{0:'b'},'logits':{0:'b'}},
        opset_version=17)
    # メタデータにペルソナ情報も含める
    meta = {
        'persona_id': persona_id,
        'persona_name': reward_params['persona_name'],
        'description': reward_params['description'],
        'reward_params': reward_params,
        'input_size': OBS_DIM,
        'output_size': ACT_DIM,
        'actions': ['forward','turn_left','turn_right'],
        'move_dist': MOVE_DIST, 'rot_deg': 20.0,
        'ray_angles_deg': RAY_DEG, 'ray_max_dist': RAY_MAX,
        'grid_size': GRID,
        'map': MAP_NP.tolist(),
        'building_cells': BCELLS_NP.tolist(),
    }
    mp = path.replace('.onnx','_meta.json')
    with open(mp,'w',encoding='utf-8') as f:
        json.dump(meta,f,indent=2,ensure_ascii=False)
    print(f'  ✓ [{persona_id}] {path}')
    policy.to(DEVICE)
    return path

print('export_persona_onnx ✓')

In [ ]:
# セル9: 報酬パラメータ読み込み
rewards_path = f'{SAVE_DIR}/persona_rewards.json'
assert os.path.exists(rewards_path), \
    f'❌ {rewards_path} が見つかりません。Step1を先に実行してください'

with open(rewards_path,encoding='utf-8') as f:
    all_rewards = json.load(f)

print(f'✓ 読み込み完了: {len(all_rewards)}ペルソナ')
for pid,p in all_rewards.items():
    print(f'  [{pid}] {p["persona_name"]}: goal={p["goal_reward"]:.1f}  '
          f'explore={p["explore_bonus"]:.1f}  social={p["social_bonus"]:.1f}')

In [ ]:
# セル10: 全ペルソナを順次学習 & ONNX生成
# ── 特定のペルソナだけ学習したい場合は TRAIN_PERSONAS を変更 ──
TRAIN_PERSONAS = list(all_rewards.keys())  # ['A','B','C','D','E']
# TRAIN_PERSONAS = ['A']  # 1つだけテストする場合

trained_policies = {}
onnx_paths = {}

for pid in TRAIN_PERSONAS:
    rp = all_rewards[pid]
    # float型に変換 (JSONから読み込んだ値がstrの場合対策)
    for k in ['explore_bonus','building_bonus','road_bonus','forward_bias',
               'revisit_penalty','violation_penalty','goal_reward',
               'step_penalty','social_bonus','stall_penalty']:
        rp[k] = float(rp.get(k, 0.0))

    policy, log = train_persona(pid, rp)
    trained_policies[pid] = policy
    path = export_persona_onnx(policy, pid, rp)
    onnx_paths[pid] = path

    # GPU メモリ解放
    del policy
    torch.cuda.empty_cache()
    print(f'\n→ [{pid}] 完了. 次のペルソナへ...\n')
    time.sleep(2)

print('='*60)
print('全ペルソナ学習完了!')
for pid,path in onnx_paths.items():
    print(f'  [{pid}] {path}')

In [ ]:
# セル11: ONNXファイルを Google Drive の専用フォルダにまとめる
import shutil

# ── 出力先フォルダ ──
# Google Drive の MyDrive/mesa_persona_onnx/ に保存します
ONNX_OUT_DIR = '/content/drive/MyDrive/mesa_persona_onnx'
os.makedirs(ONNX_OUT_DIR, exist_ok=True)
print(f'出力先: {ONNX_OUT_DIR}')
print()

copied = []
for pid in TRAIN_PERSONAS:
    onnx_src  = f'{SAVE_DIR}/persona_{pid}.onnx'
    meta_src  = f'{SAVE_DIR}/persona_{pid}_meta.json'
    onnx_dst  = f'{ONNX_OUT_DIR}/persona_{pid}.onnx'
    meta_dst  = f'{ONNX_OUT_DIR}/persona_{pid}_meta.json'

    if os.path.exists(onnx_src):
        shutil.copy2(onnx_src, onnx_dst)
        shutil.copy2(meta_src, meta_dst)
        size_mb = os.path.getsize(onnx_dst) / 1e6
        print(f'  ✓ [{pid}] persona_{pid}.onnx ({size_mb:.1f}MB)  +  _meta.json')
        copied.append(pid)
    else:
        print(f'  ⚠️  [{pid}] ファイルが見つかりません: {onnx_src}')

# persona_rewards.json もコピー
rewards_src = f'{SAVE_DIR}/persona_rewards.json'
rewards_dst = f'{ONNX_OUT_DIR}/persona_rewards.json'
if os.path.exists(rewards_src):
    shutil.copy2(rewards_src, rewards_dst)
    print(f'  ✓ persona_rewards.json')

print()
print('='*50)
print(f'✓ {len(copied)}ペルソナ分を保存しました')
print(f'  場所: マイドライブ > mesa_persona_onnx')
print()
print('フォルダ内のファイル一覧:')
for f in sorted(os.listdir(ONNX_OUT_DIR)):
    size = os.path.getsize(f'{ONNX_OUT_DIR}/{f}') / 1e6
    print(f'  {f}  ({size:.2f}MB)')
print()
print('→ persona_city_sim.html を開いて各 .onnx を読み込んでください')